# EventHallusion + VideoLLaVA on Colab

Notebook này chạy benchmark EventHallusion theo đúng mục tiêu Statement 4:

- lấy khoảng 200 video test
- kiểm chứng language prior và context bias với `misleading`
- kiểm chứng rare event xuyên suốt video với `entire`
- kiểm chứng common-rare mix với `mix`
- so sánh `normal` và `spatial_gaussian`

Output sẽ được lưu ra CSV/JSON trong thư mục kết quả để bạn đem đi phân tích.

In [ ]:
# Install dependencies
!pip -q install transformers accelerate bitsandbytes decord av huggingface_hub tqdm pandas opencv-python
!pip -q install git+https://github.com/facebookresearch/pytorchvideo.git@28fe037d212663c6a24f373b94cc5d478c8c1a1d


In [ ]:
# Clone the benchmark repo
import os
repo_dir = "/content/EventHallusion"
if not os.path.exists(repo_dir):
    !git clone https://github.com/Stevetich/EventHallusion.git /content/EventHallusion
%cd /content/EventHallusion


In [ ]:
# Optional: mount Google Drive if you want to keep videos/results there
from google.colab import drive
drive.mount('/content/drive')


## Prepare data

You need the EventHallusion videos extracted to a folder. The questions already live in `questions/` in this repo.

Typical paths:

- questions: `/content/EventHallusion/questions`
- videos: `/content/EventHallusion/videos`
- results: `/content/EventHallusion/results`

If your videos are on Drive, set `video_root` to that path instead.

In [ ]:
# Load Video-LLaVA
import torch
from transformers import VideoLlavaForConditionalGeneration, VideoLlavaProcessor

model_name = "LanguageBind/Video-LLaVA-7B-hf"
processor = VideoLlavaProcessor.from_pretrained(model_name)
model = VideoLlavaForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)
print("Model loaded")


In [ ]:
# Run EventHallusion benchmark
from eventhallusion_videollava_eval import compare_conditions

questions_root = "/content/EventHallusion/questions"
video_root = "/content/EventHallusion/videos"
out_dir = "/content/EventHallusion/results"

# 200 videos total, split evenly across the three EventHallusion groups
summary = compare_conditions(
    model=model,
    processor=processor,
    questions_root=questions_root,
    video_root=video_root,
    out_dir=out_dir,
    n_total_videos=200,
    n_frames=8,
    sigma=25,
    seed=42,
    per_split={"misleading": 67, "entire": 67, "mix": 66},
)

summary


## What to look at

- `acc_misleading`: language prior / context bias
- `acc_entire`: whether the model follows the whole video timeline
- `acc_mix`: whether the model can avoid frame-level shortcut on mixed rare/common events

Compare `normal` vs `spatial_gaussian`.
If spatial Gaussian helps mostly on `misleading`, then it mainly attacks spatial/context shortcuts rather than temporal reasoning itself.

In [ ]:
# Download results to Drive if needed
import shutil
drive_out = "/content/drive/MyDrive/EventHallusion_results"
os.makedirs(drive_out, exist_ok=True)
if os.path.exists(out_dir):
    for name in [
        "eventhallusion_summary.csv",
        "eventhallusion_normal.csv",
        "eventhallusion_spatial_gaussian.csv",
        "eventhallusion_normal_predictions.json",
        "eventhallusion_spatial_gaussian_predictions.json",
    ]:
        src = os.path.join(out_dir, name)
        if os.path.exists(src):
            shutil.copy(src, os.path.join(drive_out, name))
print(drive_out)
